# Day 5 — Explore Chunking Strategies

Compare the two chunkers (fixed-size and paragraph-aware) on real
Wikipedia articles. Look at chunk boundaries, readability, and size
distributions.

In [ ]:
import sys
sys.path.insert(0, "../src")

from pathlib import Path
import pyarrow.parquet as pq

from workbench.data_build.chunking import (
    FixedSizeChunker,
    ParagraphAwareChunker,
)

## Load a few sample articles

In [ ]:
articles_path = Path("../data/processed/wikipedia_articles.parquet")
table = pq.read_table(articles_path)
df = table.to_pandas()

# Pick 3 articles of varying lengths
sample_titles = ["Earth", "Oxygen", "Python (programming language)"]
samples = df[df["title"].isin(sample_titles)].copy()

# If exact matches aren't found, fall back to the first 3 articles > 2000 chars
if len(samples) < 2:
    samples = df[df["text_length"] > 2000].head(3).copy()

print(f"Sample articles: {len(samples)}")
for _, row in samples.iterrows():
    print(f"  {row['title']:40s}  ({row['text_length']:,} chars)")

## Set up both chunkers

In [ ]:
fixed = FixedSizeChunker(chunk_size=1000, overlap=200, min_chunk_size=100)
para  = ParagraphAwareChunker(target_size=1000, max_size=1500, min_chunk_size=100)

print(f"Fixed-size:       {fixed.settings_str}")
print(f"Paragraph-aware:  {para.settings_str}")

## Compare: first 5 chunks from each chunker

Look at how each chunker splits the same article. The paragraph-aware
chunker should produce chunks that start and end at natural boundaries.

In [ ]:
for _, row in samples.iterrows():
    doc_id = row["page_id"]
    title = row["title"]
    text = row["text"]

    chunks_fixed = fixed.chunk(doc_id, title, text)
    chunks_para = para.chunk(doc_id, title, text)

    print("=" * 80)
    print(f"ARTICLE: {title} ({len(text):,} chars)")
    print(f"  Fixed-size chunks: {len(chunks_fixed)},  "
          f"Paragraph-aware chunks: {len(chunks_para)}")
    print()

    for label, chunks in [("FIXED-SIZE", chunks_fixed), ("PARAGRAPH-AWARE", chunks_para)]:
        print(f"  --- {label} (first 3 chunks) ---")
        for i, c in enumerate(chunks[:3]):
            preview = c.text[:150].replace("\n", " ")
            print(f"  [{i}] chars {c.start_char}-{c.end_char} ({len(c.text)} chars)")
            print(f"      {preview}...")
            print()
    print()

## Chunk size distributions

In [ ]:
# Chunk all sample articles with both chunkers and compare size distributions
all_fixed_lengths = []
all_para_lengths = []

# Use a larger sample for the distribution
bigger_sample = df[df["text_length"] > 500].sample(min(500, len(df)), random_state=42)

for _, row in bigger_sample.iterrows():
    for c in fixed.chunk(row["page_id"], row["title"], row["text"]):
        all_fixed_lengths.append(len(c.text))
    for c in para.chunk(row["page_id"], row["title"], row["text"]):
        all_para_lengths.append(len(c.text))

print(f"Fixed-size:  {len(all_fixed_lengths):,} chunks from {len(bigger_sample)} articles")
print(f"  min={min(all_fixed_lengths)}, max={max(all_fixed_lengths)}, "
      f"mean={sum(all_fixed_lengths)/len(all_fixed_lengths):.0f}")
print()
print(f"Paragraph-aware: {len(all_para_lengths):,} chunks from {len(bigger_sample)} articles")
print(f"  min={min(all_para_lengths)}, max={max(all_para_lengths)}, "
      f"mean={sum(all_para_lengths)/len(all_para_lengths):.0f}")

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].hist(all_fixed_lengths, bins=50, edgecolor="white", alpha=0.8)
    axes[0].set_title(f"Fixed-size ({fixed.settings_str})")
    axes[0].set_xlabel("Chunk length (chars)")
    axes[0].set_ylabel("Count")

    axes[1].hist(all_para_lengths, bins=50, edgecolor="white", alpha=0.8, color="orange")
    axes[1].set_title(f"Paragraph-aware ({para.settings_str})")
    axes[1].set_xlabel("Chunk length (chars)")
    axes[1].set_ylabel("Count")

    plt.tight_layout()
    plt.show()
except ImportError:
    print("pip install matplotlib for plots")

## Check chunk ID stability

Run the same chunker twice and verify IDs match.

In [ ]:
# Pick one article
test_row = bigger_sample.iloc[0]

run_a = para.chunk(test_row["page_id"], test_row["title"], test_row["text"])
run_b = para.chunk(test_row["page_id"], test_row["title"], test_row["text"])

ids_match = all(a.chunk_id == b.chunk_id for a, b in zip(run_a, run_b))
print(f"Article: {test_row['title']}")
print(f"Chunks run A: {len(run_a)}, run B: {len(run_b)}")
print(f"All IDs match: {ids_match}  ✓" if ids_match else "IDs MISMATCH ✗")

## Inspect the full chunks.parquet (after running build_chunks.py)

In [ ]:
chunks_path = Path("../data/processed/chunks.parquet")
if chunks_path.exists():
    chunks_table = pq.read_table(chunks_path)
    chunks_df = chunks_table.to_pandas()
    print(f"Total chunks: {len(chunks_df):,}")
    print(f"Columns: {list(chunks_df.columns)}")
    print(f"Chunker: {chunks_df['chunker_name'].iloc[0]} ({chunks_df['chunker_settings'].iloc[0]})")
    print()
    print("Text length stats:")
    print(chunks_df["text_length"].describe().to_string())
    print()
    print("First 5 chunks:")
    for _, row in chunks_df.head(5).iterrows():
        preview = row["text"][:100].replace("\n", " ")
        print(f"  [{row['chunk_id'][:8]}] {row['title'][:30]:30s} "
              f"({row['text_length']} chars): {preview}...")
else:
    print(f"chunks.parquet not found at {chunks_path}")
    print("Run: python scripts/build_chunks.py")

## Summary

Key observations to look for:
- **Fixed-size** chunks have very uniform length but may cut mid-sentence
- **Paragraph-aware** chunks vary more in size but read more naturally
- Both produce stable, deterministic chunk IDs
- The paragraph-aware chunker is generally better for RAG because LLMs
  benefit from coherent context passages